In [3]:
import os
import math 
import time
import requests
import torch
import torch.nn as nn #
import torch.nn.functional as F

In [5]:
data_url= "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [7]:
data_path="data/input.txt"
ckpt_path="checkpoints/model.pt"

In [11]:
block_size=128
batch_size=32
max_iter=3000
eval_every=300
eval_iters=200
lr=3e-4
n_embed=128
n_head=4
n_layer=4
dropout=0.1

device="cuba" if torch.cuda.is_available() else "cpu"

In [12]:
print(f"using device:{device}")

using device:cpu


In [15]:
def download_data():
    os.makedirs("data",exist_ok=True)
    if not os.path.exists(data_path):
        print("downloading shakespeare dataset")
        r=requests.get(data_url)
        with open(data_path,"w",encoding="utf-8") as f:
            f.write(r.text)
        print(f"Saved to {data_path} ({len(r.text):,} chars)")
    else:
        print(f"data already exists at {data_path} ")
    

download_data()

downloading shakespeare dataset
Saved to data/input.txt (1,115,394 chars)


In [16]:
with open(data_path,"r",encoding="utf-8") as f:
    text=f.read()
print(f"length of dataset in characters: {len(text):}")

length of dataset in characters: 1115394


2.TOKENIZER

In [17]:
chars=sorted(set(text))
vocab_size=len(chars)
print(f"unique characters: {vocab_size}")
print(f"vocab: {''.join(chars)}")

unique characters: 65
vocab: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [19]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[stoi[c] for c in s]
decode=lambda l:"".join([itos[i] for i in l])


3.PREPARE DATA

In [20]:
data=torch.tensor(encode(text),dtype=torch.long)
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]
print(f"train has {len(train_data):,} tokens, val has {len(val_data):,} tokens")

train has 1,003,854 tokens, val has 111,540 tokens


In [21]:
def get_batch(split):
    d=train_data if split=="train" else val_data
    ix=torch.randint(len(d)-block_size,(batch_size,))
    x=torch.stack([d[i:i+block_size] for i in ix])
    y=torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device),y.to(device)

4.MODEL ARCHITECTURE

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C=x.shape
        k=self.key(x)  
        q=self.query(x)
        v=self.value(x)
        
        scale = k.shape[-1]**-0.5
        attn=q @ k.transpose(-2,-1)*scale 

        attn=attn.masked_fill(self.tril[:T,:T]==0,float("-inf"))
        attn=F.softmax(attn,dim=-1)
        attn=self.dropout(attn)
        
        out=attn @ v 
        return out

In [24]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [25]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [26]:
class TransformerBlock(nn.Module):
    """ a single transformer block """

    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.attn = MultiHeadAttention(n_head, head_size)
        self.ff = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [27]:
class GPT(nn.Module):
    """ the full GPT language model, with a context size of block_size """

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embed, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed) # final layer norm
        self.lm_head = nn.Linear(n_embed, vocab_size)

        self.apply(self._init_weights)

    def forward(self, idx, targets=None):
        B,T=idx.shape

        tok_emb=self.token_embedding_table(idx) # (B,T,C)
        pos_emb=self.position_embedding_table(torch.arange(T,device=device)) # (T,C)
        x=tok_emb+pos_emb # (B,T,C)
        x=self.blocks(x) # (B,T,C)
        x=self.ln_f(x) # (B,T,C)
        logits=self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)

        return logits,loss

    @torch.no_grad()
    def generate(self, idx,max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond=idx[:,-block_size:]
            logits,_=self(idx_cond)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            next_tok=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,next_tok),dim=1)
        return idx
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    

5.  INIT MODEL +OPTIMISER

In [30]:
model=GPT().to(device)
num_params=sum(p.numel() for p in model.parameters())
print(f"number of parameters: {num_params:,}")

optimizer=torch.optim.AdamW(model.parameters(),lr=lr)

number of parameters: 824,897


6.EVALUATION HELPER

In [36]:
@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ["train","val"]:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb,yb=get_batch(split)
            _,loss=model(xb,yb)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

7.TRAINING LOOP

In [37]:
print("\n"+"="*55)

os.makedirs("checkpoints",exist_ok=True)
best_val_loss=float("inf")
t0=time.time()

for step in range(max_iter):
    if step % eval_every==0 or step==max_iter-1:
        losses=estimate_loss()
        elapsed=time.time()-t0
        print(f"step {step:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, time elapsed {elapsed:.1f} seconds")

        if losses["val"]<best_val_loss:
            best_val_loss=losses["val"]
            torch.save({"step":step,"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),"val_loss":best_val_loss,"vocab_size":vocab_size,"stoi":stoi,"itos":itos},ckpt_path)

    xb,yb=get_batch("train")
    logits,loss=model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()

print("training complete")


step    0: train loss 4.1860, val loss 4.1889, time elapsed 65.9 seconds
step  300: train loss 2.4009, val loss 2.4067, time elapsed 415.0 seconds
step  600: train loss 2.1344, val loss 2.1698, time elapsed 767.2 seconds
step  900: train loss 1.8761, val loss 1.9701, time elapsed 1126.1 seconds
step 1200: train loss 1.7352, val loss 1.8828, time elapsed 1490.9 seconds
step 1500: train loss 1.6415, val loss 1.8185, time elapsed 1899.3 seconds
step 1800: train loss 1.5875, val loss 1.7649, time elapsed 2331.4 seconds
step 2100: train loss 1.5336, val loss 1.7320, time elapsed 2780.5 seconds
step 2400: train loss 1.4989, val loss 1.6986, time elapsed 3253.1 seconds
step 2700: train loss 1.4656, val loss 1.6659, time elapsed 3904.5 seconds
step 2999: train loss 1.4559, val loss 1.6574, time elapsed 4231.9 seconds
training complete


8. GENERATE TEXT FROM TRAINED MODEL

In [43]:
print("creating model")
model.eval()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500)
generated_text = decode(generated[0].tolist())
 
print(generated_text)
print("="*55)

creating model


FRIAR LAURENCE:
Why be for Harn ne's father rison?

SITRES OF YORK:
Were make the battle into his lain,
Do beg'd, looks now undead as encan thy Romeo,
A wiftediouse of power's well; more kich I know?
I knock'd Have cain'd it by an friends: your is igland's appart
To heavy notling gotle of you the simpart of a liber
Of carel my favour 'Will mind near, I slaike untied your
Or faul lords. Son, I say, I not let in tyrand grief.

TAHASLILO:
O more: his thou do: see you have on, you'll eprison; if th
